
# 02 &middot; Validation

**~50 minutes. Strongly recommended &mdash; more than any card.**

You get six leaderboard submissions today. So the number that actually governs
your day is not your test score, it is your **estimate** of your test score.
This notebook is about making that estimate trustworthy.

The one thing to get straight before you start:

> You are not looking for the split that gives you the best score.
> You are looking for the split whose score you would **bet money** matches
> your test score.

Those pull in opposite directions, and the pull toward the flattering one is
strong. Watch for it.

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn

# Get common.py. If you uploaded it yourself (folder icon in the left sidebar),
# this leaves your copy alone -- it only downloads when the file is missing.
!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/CHANGE-ME/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="CHANGE-ME")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from lightgbm import LGBMRegressor
sns.set_style("whitegrid")

train = common.load_train()
test  = common.load_test()

# One fixed model, reused for every experiment below. We are comparing
# SPLITS, so the model must be held constant.
X_all = common.rdkit_descriptors(train[common.SMILES_COL])
print(X_all.shape, "descriptors")

In [ ]:
def fit_predict(fold, X=X_all, df=train, endpoints=None):
    """Train on fold=='train', predict fold=='val'. One model per endpoint."""
    endpoints = endpoints or common.ENDPOINTS
    tr, va = (fold == "train").to_numpy(), (fold == "val").to_numpy()
    Xtr, Xva = common.clean_features(X[tr], X[va])
    out = pd.DataFrame({common.ID_COL: df.loc[va, common.ID_COL].to_numpy()})
    for e in endpoints:
        y = df.loc[tr, e]
        ok = y.notna().to_numpy()
        if ok.sum() < 50:
            out[e] = np.nan; continue
        m = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                          num_leaves=31, verbose=-1, n_jobs=-1)
        m.fit(Xtr[ok], y[ok])
        out[e] = m.predict(Xva)
    return out, df.loc[va].reset_index(drop=True)

---
## 1. Four splits, one model

- **random** &mdash; shuffle and cut
- **scaffold** &mdash; whole Bemis-Murcko scaffold groups go to one side, so
  validation contains chemical series the model has never seen
- **temporal** &mdash; the last-registered compounds are held out, mimicking the
  real train/test split
- **similarity** &mdash; hold out the molecules least similar to everything else

A note on how temporal works here: there is no date column. But compound IDs
look like `E-0001321`, and registration numbers increase over time, so the
number is a usable proxy for synthesis date. Improvising a time axis out of an
ID scheme is an extremely normal thing to have to do with real data.

### &#9654;&#65039; Predict first

**Rank the four splits from the one that will give the BEST-looking score to the one that will give the worst. Then say which one you think will be closest to the real test score.**

*Those two answers should not be the same split. If they are, think again.*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

> The next cell trains 9 models &times; 4 splits and computes an
> all-against-all similarity matrix. **Expect 5&ndash;10 minutes.** It has not
> hung. Good time to write down your prediction above, or to start reading
> `REPORT_TEMPLATE.md`.

In [ ]:
results = {}
preds = {}
for name, fn in common.SPLITTERS.items():
    fold = fn(train)
    p, truth = fit_predict(fold)
    ev = common.evaluate(truth, p)
    results[name] = ev
    preds[name] = (p, truth, fold)
    print(f"{name:11s} val MA-RAE = {ev['RAE'].mean():.3f}   "
          f"({(fold=='val').sum()} molecules held out)")

In [ ]:
comp = pd.DataFrame({k: v["RAE"] for k, v in results.items()})
fig, ax = plt.subplots(figsize=(10, 4.5))
comp.plot(kind="bar", ax=ax)
ax.axhline(1.0, color="k", ls="--", lw=1, label="guessing the mean")
ax.set_ylabel("RAE (lower is better)")
ax.set_title("Same model, same data, four different answers")
ax.legend(); plt.tight_layout(); plt.show()
comp.round(3)

### What just happened

One model. One dataset. Four numbers, and they are not close.

The random split is almost always the most flattering, because random holdout
molecules have close analogues sitting in the training set &mdash; often the same
scaffold with one substituent changed. The model does not have to generalise;
it has to interpolate between near-twins.

**A validation split is a hypothesis about how the model will be used.** If you
will only ever predict compounds closely related to what you have measured, the
random split is honest. If you will predict next month's designs, it is a lie.

Here, the test set is late-stage compounds from the same campaign. Which of
your four numbers do you now believe?

---
## 2. Metrics disagree with each other

Everything so far has been RAE. Now look at the same predictions through
different lenses.

In [ ]:
ev = results["temporal"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ev[["MAE", "RAE"]].plot(kind="bar", ax=axes[0], title="error-based")
ev[["R2", "Spearman", "Kendall"]].plot(kind="bar", ax=axes[1], title="rank-based")
axes[1].axhline(0, color="k", lw=1)
plt.tight_layout(); plt.show()
ev.round(3)

### Find the endpoint that is lying to you

At least one endpoint here has a **respectable MAE and a poor Kendall's tau**.
That combination means: the model gets the magnitude roughly right and the
*ordering* wrong.

Which matters more? Depends entirely on the question. A medicinal chemist
holding twenty proposed compounds does not need to know that compound 7 has
log solubility -4.6. They need to know that compound 7 is more soluble than
compound 12. **That is a ranking question, and MAE does not measure it.**

In the real challenge this was KSOL's signature failure. The data is
bimodal &mdash; most compounds are either above 200 &micro;M or below 5 &micro;M &mdash;
so a model can score decently on error by learning "high pile or low pile"
while being unable to order anything within a pile. It had become a classifier
wearing a regressor's clothes. Exactly the same behaviour showed up in the
ASAP Discovery / Polaris antiviral challenge.

Plot it and see.

In [ ]:
p, truth, _ = preds["temporal"]
merged = truth.merge(p, on=common.ID_COL, suffixes=("_true", "_pred"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, e in zip(axes, ["LogD", "LogS", "Log_HLM_CLint"]):
    a, b = merged[e + "_true"], merged[e + "_pred"]
    ok = a.notna() & b.notna()
    ax.scatter(a[ok], b[ok], s=8, alpha=.35)
    lo, hi = np.nanpercentile(a[ok], [1, 99])
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    r = ev.loc[e]
    ax.set_title(f"{e}\nMAE={r['MAE']:.2f}  tau={r['Kendall']:.2f}", fontsize=10)
    ax.set_xlabel("measured"); ax.set_ylabel("predicted")
plt.tight_layout(); plt.show()

Look at the shape of the LogS cloud versus the LogD cloud. LogD
scatters along the diagonal. LogS collapses toward the extremes &mdash; the model
is pushing predictions to the two piles rather than tracking structure.

---
## 3. Where should you *not* trust this model?

Take the predictions you already have, bucket the validation molecules by how
similar they are to the training set, and look at the error in each bucket.

No retraining. One plot. This works on any predictions you ever make, which
makes it the cheapest useful thing in this notebook.

In [ ]:
p, truth, fold = preds["temporal"]
fp_all = common.morgan_fingerprints(train[common.SMILES_COL])
tr_idx = np.where((fold == "train").to_numpy())[0]
va_idx = np.where((fold == "val").to_numpy())[0]

nn = common.nearest_neighbour_similarity([fp_all[i] for i in va_idx],
                                         [fp_all[i] for i in tr_idx])
m = truth.merge(p, on=common.ID_COL, suffixes=("_true", "_pred"))
m["nn_sim"] = nn
m["bucket"] = pd.qcut(m["nn_sim"], 4, duplicates="drop")

rows = []
for b, g in m.groupby("bucket", observed=True):
    err = np.concatenate([(g[e + "_true"] - g[e + "_pred"]).abs().dropna().to_numpy()
                          for e in common.ENDPOINTS])
    rows.append({"similarity bucket": str(b), "n errors": len(err),
                 "mean |error|": err.mean()})
ad = pd.DataFrame(rows)
ad

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(ad)), ad["mean |error|"], color="#468")
ax.set_xticks(range(len(ad)))
ax.set_xticklabels(ad["similarity bucket"], rotation=20, fontsize=8)
ax.set_xlabel("Tanimoto similarity to nearest training molecule")
ax.set_ylabel("mean absolute error")
ax.set_title("Applicability domain: error vs. how familiar the molecule is")
plt.tight_layout(); plt.show()

**Write down the number.** Something like *"below Tanimoto 0.4 we
would not trust this model"* is an **applicability domain** &mdash; a decision
rule for when to believe your own predictions. It goes on slide 6.

**If the effect is weak, that is the more interesting result.** It would mean
this test set sits comfortably inside the training set's chemical space: the
task is *interpolation*, which is exactly why it is tractable at all. Contrast
that with the public zero-shot models from `01_eda`, which were attacking the
same molecules from below the similarity noise floor and produced negative
R&sup2;.

Same molecules, same metric, opposite outcome &mdash; the only difference is
whether the training data was in the neighbourhood.

---
## 4. Invent your own split

The four built-ins are not the only options, and in the real challenge one of
the top-5 finishers (*shin-chan*) used something else entirely: a
**difficulty-based** split, holding out the molecules the model found hardest,
to force honest validation against awkward chemistry. Another top-20 finisher
used a similarity-based split.

Write a function that returns a Series of `"train"`/`"val"` for each row.

Ideas: hold out one scaffold family entirely; hold out the most lipophilic
decile; hold out compounds with the most missing data; cluster the fingerprints
and hold out whole clusters.

In [ ]:
def my_split(df):
    # YOUR CODE. Return a pd.Series of "train"/"val", one per row of df.
    raise NotImplementedError

# fold = my_split(train)
# p, truth = fit_predict(fold)
# print("MA-RAE:", common.evaluate(truth, p)["RAE"].mean().round(3))

---
## 5. Commit

Pick the split you will use for the rest of the day and save it. Every other
notebook will pick it up automatically via `common.load_split()` &mdash; you do
not need to run them in any order, and you do not need to copy anything.

Choose on **trustworthiness, not on score.**

In [ ]:
CHOICE = "temporal"          # <-- your pick
RATIONALE = "..."            # <-- one sentence you would defend out loud

fold = common.SPLITTERS[CHOICE](train)
val_score = results[CHOICE]["RAE"].mean()

common.save_split(fold, train, method=CHOICE,
                  rationale=RATIONALE, val_score=float(val_score))
print(f"\nyour validation estimate: MA-RAE = {val_score:.3f}")

### The calibration leaderboard

That number is now your public prediction. When the test scores land this
afternoon, there is a second leaderboard ranking pairs on

$$|\text{your validation estimate} - \text{your actual test score}|$$

Note what it rewards: not being *good*, but knowing how good you are. A pair
with MA-RAE 0.75 who predicted 0.75 beats a pair with 0.60 who predicted 0.45.

This is also why picking the flattering split backfires. Random will hand you
a lovely 0.55 and then the test will come in at 0.72, and everyone will see the
gap.

---
## Baseline for the rest of the day

Save these predictions so the cards have something to be compared against, and
so `card_ensembles` has something to work with.

In [ ]:
fold_final, _ = common.load_split(train)
X_test = common.rdkit_descriptors(test[common.SMILES_COL])
Xtr, Xte = common.clean_features(X_all, X_test)

pred = common.blank_predictions(test)
for e in common.ENDPOINTS:
    y = train[e]; ok = y.notna().to_numpy()
    m = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                      num_leaves=31, verbose=-1, n_jobs=-1)
    m.fit(Xtr[ok], y[ok])
    pred[e] = m.predict(Xte)

common.save_predictions(pred, "lgbm-rdkit-baseline",
                        note="LightGBM on RDKit descriptors, one model per endpoint")

This is the reference baseline &mdash; the same recipe OpenADMET used
as their own comparison model. If a fancier approach does not beat it, the
fancier approach is not working, and finding that out costs you ten minutes
instead of three hours.

**Do not submit yet.** You have six shots. Go take a card first.
&rarr; `README.md`